In [29]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import torch.optim.lr_scheduler as lr_scheduler

In [2]:
class CustomImageDataset(Dataset):
    def __init__(self, csv_file, transform=None):
       
        self.data_frame = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_path = self.data_frame.iloc[idx]['image_path']
        label = int(self.data_frame.iloc[idx]['class_id'])

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

In [10]:
train_mean = [0.8738403271227945, 0.7486465899118838, 0.7215444216680992]
train_std = [0.15869203339064175, 0.185724253225529, 0.07973004670164234]

train_transform = transforms.Compose([
    transforms.Resize((360, 360)),

    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.1),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),

    transforms.ToTensor(),
    transforms.Normalize(mean=train_mean, std=train_std)
])

val_test_transform = transforms.Compose([
    transforms.Resize((360, 360)),
    transforms.ToTensor(),
    transforms.Normalize(mean=train_mean, std=train_std)
])

In [11]:
train_dataset = CustomImageDataset(csv_file='train_split.csv', transform=train_transform)
val_dataset = CustomImageDataset(csv_file='val_split.csv', transform=val_test_transform)
test_dataset = CustomImageDataset(csv_file='test_split.csv', transform=val_test_transform)

In [12]:
BATCH_SIZE = 8
NUM_WORKERS = 4

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,       
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,        
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,     
    num_workers=NUM_WORKERS,
)

In [15]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=360, patch_size=20, in_channels=3, emb_dim=128):
        super().__init__()

        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches_per_dim = img_size // patch_size
        self.num_patches = self.num_patches_per_dim ** 2

        self.proj = nn.Conv2d(
            in_channels=in_channels,
            out_channels=emb_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        x = self.proj(x)            
        x = x.flatten(2)             
        x = x.transpose(1, 2)        
        return x

In [16]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        img_size=360,
        patch_size=20,
        in_channels=3,
        num_classes=8,
        emb_dim=128,
        depth=4,
        num_heads=4,
        mlp_dim=256,
        dropout=0.1
    ):
        super().__init__()

        self.patch_embed = PatchEmbedding(
            img_size=img_size,
            patch_size=patch_size,
            in_channels=in_channels,
            emb_dim=emb_dim
        )

        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.randn(1, 1, emb_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, emb_dim))
        self.dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim,
            nhead=num_heads,
            dim_feedforward=mlp_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=depth
        )

        self.norm = nn.LayerNorm(emb_dim)
        self.classifier = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]

        x = self.patch_embed(x)   

        cls_token = self.cls_token.expand(B, -1, -1)  
        x = torch.cat([cls_token, x], dim=1) 

        x = x + self.pos_embed
        x = self.dropout(x)

        x = self.encoder(x)      
        x = self.norm(x)

        cls_output = x[:, 0]     
        logits = self.classifier(cls_output)  

        return logits

In [17]:
model = VisionTransformer(
        img_size=360,
        patch_size=20,
        in_channels=3,
        num_classes=8, 
        emb_dim=128,
        depth=4,
        num_heads=4,
        mlp_dim=256,
        dropout=0.1
    )

x = torch.randn(8, 3, 360, 360)
y = model(x)

print("Input shape :", x.shape)
print("Output shape:", y.shape)

C:\Users\Ayeezh\AppData\Local\Temp\ipykernel_12008\2076479833.py:39: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


Input shape : torch.Size([8, 3, 360, 360])
Output shape: torch.Size([8, 8])


In [23]:
train_df = pd.read_csv('train_split.csv')

class_counts = train_df['class_id'].value_counts().sort_index().values
class_counts = torch.tensor(class_counts, dtype=torch.float32)

class_weights = class_counts.sum() / (len(class_counts) * class_counts)

In [24]:
class_weights

tensor([1.7547, 0.6853, 1.3772, 0.7380, 1.7602, 1.5045, 0.6418, 0.9101])

In [26]:
NUM_CLASSES = 8
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [28]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs,
    save_path="best_model.pt",
    scheduler=None
):

    model = model.to(device)
    best_val_loss = float("inf")

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "val_precision": [],
        "val_recall": [],
        "val_f1": [],
        
        "train_batch_loss": [],
        "train_batch_acc": [],
        "val_batch_loss": [],
        "val_batch_acc": [],
        
        "lr": [],
        "best_val_loss": [],
        "last_val_preds": None,
        "last_val_targets": None
    }

    for epoch in range(num_epochs):

        model.train()
        running_train_loss = 0.0
        running_train_correct = 0
        running_train_total = 0

        train_bar = tqdm(
            train_loader,
            desc=f"Epoch [{epoch+1}/{num_epochs}] - Training",
            leave=True
        )

        for batch_idx, (images, labels) in enumerate(train_bar):
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            preds = torch.argmax(outputs, dim=1)
            batch_size = labels.size(0)
            batch_correct = (preds == labels).sum().item()
            batch_acc = batch_correct / batch_size

            history["train_batch_loss"].append(loss.item())
            history["train_batch_acc"].append(batch_acc)

            running_train_loss += loss.item() * batch_size
            running_train_correct += batch_correct
            running_train_total += batch_size

            current_train_loss = running_train_loss / running_train_total
            current_train_acc = running_train_correct / running_train_total

            train_bar.set_postfix({
                "batch_loss": f"{loss.item():.4f}",
                "batch_acc": f"{batch_acc:.4f}",
                "train_loss": f"{current_train_loss:.4f}",
                "train_acc": f"{current_train_acc:.4f}",
                "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
            })

        epoch_train_loss = running_train_loss / running_train_total
        epoch_train_acc = running_train_correct / running_train_total

        history["train_loss"].append(epoch_train_loss)
        history["train_acc"].append(epoch_train_acc)


        model.eval()
        running_val_loss = 0.0
        running_val_correct = 0
        running_val_total = 0

        all_val_preds = []
        all_val_targets = []

        val_bar = tqdm(
            val_loader,
            desc=f"Epoch [{epoch+1}/{num_epochs}] - Validation",
            leave=True
        )

        with torch.no_grad():
            for batch_idx, (images, labels) in enumerate(val_bar):
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                preds = torch.argmax(outputs, dim=1)
                
                all_val_preds.extend(preds.cpu().numpy())
                all_val_targets.extend(labels.cpu().numpy())

                batch_size = labels.size(0)
                batch_correct = (preds == labels).sum().item()
                batch_acc = batch_correct / batch_size

                history["val_batch_loss"].append(loss.item())
                history["val_batch_acc"].append(batch_acc)

                running_val_loss += loss.item() * batch_size
                running_val_correct += batch_correct
                running_val_total += batch_size

                current_val_loss = running_val_loss / running_val_total
                current_val_acc = running_val_correct / running_val_total

                val_bar.set_postfix({
                    "batch_loss": f"{loss.item():.4f}",
                    "batch_acc": f"{batch_acc:.4f}",
                    "val_loss": f"{current_val_loss:.4f}",
                    "val_acc": f"{current_val_acc:.4f}",
                })

        epoch_val_loss = running_val_loss / running_val_total
        epoch_val_acc = running_val_correct / running_val_total


        val_precision, val_recall, val_f1, _ = precision_recall_fscore_support(
            all_val_targets, 
            all_val_preds, 
            average='macro', 
            zero_division=0
        )

        history["val_loss"].append(epoch_val_loss)
        history["val_acc"].append(epoch_val_acc)
        history["val_precision"].append(val_precision)
        history["val_recall"].append(val_recall)
        history["val_f1"].append(val_f1)
        
        history["last_val_preds"] = all_val_preds
        history["last_val_targets"] = all_val_targets

        current_lr = optimizer.param_groups[0]["lr"]
        history["lr"].append(current_lr)


        if scheduler is not None:
            if scheduler.__class__.__name__ == "ReduceLROnPlateau":
                scheduler.step(epoch_val_loss)
            else:
                scheduler.step()


        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss

            checkpoint = {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "train_loss": epoch_train_loss,
                "train_acc": epoch_train_acc,
                "val_loss": epoch_val_loss,
                "val_acc": epoch_val_acc,
                "val_f1": val_f1
            }

            if scheduler is not None:
                checkpoint["scheduler_state_dict"] = scheduler.state_dict()

            torch.save(checkpoint, save_path)
            save_status = f"✅ Best model saved/overridden at: {save_path}"
        else:
            save_status = "— No improvement in Val Loss"

        history["best_val_loss"].append(best_val_loss)


        print(f"\n=================== Epoch [{epoch+1}/{num_epochs}] Summary ===================")
        print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
        print(f"Val   Loss: {epoch_val_loss:.4f} | Val   Acc: {epoch_val_acc:.4f}")
        print(f"Val F1-Score (Macro): {val_f1:.4f} | Val Precision: {val_precision:.4f} | Val Recall: {val_recall:.4f}")
        print(f"Best Val Loss: {best_val_loss:.4f} | LR: {current_lr:.2e}")
        print(save_status)
        print("=" * 60 + "\n")

    return history


In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VisionTransformer(
        img_size=360,
        patch_size=20,
        in_channels=3,
        num_classes=8, 
        emb_dim=128,
        depth=4,
        num_heads=4,
        mlp_dim=256,
        dropout=0.1
    )

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)


C:\Users\Ayeezh\AppData\Local\Temp\ipykernel_12008\2076479833.py:39: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(


In [ ]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=20,
    save_path="best_vit_model.pt",
    scheduler=scheduler
)